In [2]:
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project='osmgee')

watersheds = ee.FeatureCollection('projects/osmgee/assets/OSM_Watersheds')
watershed_geom = watersheds.geometry()

ASSET_PROJECT = 'projects/annual-ccdc-new/assets'

FIVE_YEAR_YEARS = list(range(1985, 2026, 5))
print('Years to load:', FIVE_YEAR_YEARS)

Years to load: [1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025]


In [3]:
# ============================================
# NEW: Load the 7-class ABMI photoplot training source
# ============================================

training_source = ee.FeatureCollection(
    'projects/annual-ccdc-new/assets/OSMWetland/ABMI_PhotoPlots_Clipped_Watershed'
)

print('Feature count:', training_source.size().getInfo())
print('Properties of first feature:', training_source.first().propertyNames().getInfo())

Feature count: 442
Properties of first feature: ['UHEIGHT', 'NV_PER', 'UORIGIN_YR', 'MOD3_PER', 'INFRA_TY', 'SP4_PER', 'WTLD_TY', 'HYDR_REG', 'DENSITY', 'LU1_LEVEL1', 'MyClass', 'LU1_LEVEL2', 'LU2_LEVEL2', 'STATUS', 'MPT_CNT', 'LU2_LEVEL1', 'SP3_PER', 'NV_TYPE', 'MOIST_REG', 'LC2', 'LC1', 'NWOOD_PER', 'LC3', 'AREA_NET', 'USP5', 'USP4', 'SP2_PER', 'USP3', 'GAWL_TY', 'USP2', 'INFRA_CL', 'MOD1', 'MOD1_YR', 'NUTR_REG', 'WAWL_TY', 'USITE_HT', 'MOD2', 'SP1_PER', 'USP4_PER', 'MOD3', 'NV_CLASS', 'USP5_PER', 'STAND_STRU', 'MOD2_YR', 'UDENSITY', 'NTW_TY', 'ORIGIN', 'SEC_HT', 'USP1', 'USP3_PER', 'NTW_HT', 'WAUL_TY', 'SOIL_TY', 'QC_ATTR3B', 'UORIGIN', 'OBS', 'MOD1_PER', 'MOD3_YR', 'ORIGIN_YR', 'USP2_PER', 'ABMI_SITE', 'NWOOD_TY', 'HEIGHT', 'Shape_Leng', 'SITE_HT', 'MOD2_PER', 'SP2', 'SP1', 'SP4', 'SP5_PER', 'USP1_PER', 'SP3', 'PER_POLY', 'Shape_Area', 'SP5', 'POLYGON_ID', 'WetlandTyp', 'NTW_PER', 'system:index']


In [4]:
# ============================================
# STEP 2: Confirm the 7-class distribution in WetlandTyp
# ============================================

print('WetlandTyp values:', training_source.aggregate_array('WetlandTyp').distinct().getInfo())
print('Class distribution:', training_source.aggregate_histogram('WetlandTyp').getInfo())

WetlandTyp values: ['Fen', 'Marsh', 'Shrub', 'Open water', 'Swamp', 'Forest', 'Bog']
Class distribution: {'Bog': 12, 'Fen': 96, 'Forest': 274, 'Marsh': 3, 'Open water': 5, 'Shrub': 10, 'Swamp': 42}


In [5]:
# ============================================
# STEP 3: Check polygon area distribution, all 7 classes
# ============================================

areas_and_classes = training_source.reduceColumns(
    reducer=ee.Reducer.toList(2),
    selectors=['Shape_Area', 'WetlandTyp']
).get('list').getInfo()

df_areas = pd.DataFrame(areas_and_classes, columns=['Shape_Area', 'WetlandTyp'])

print(df_areas.groupby('WetlandTyp')['Shape_Area'].describe())

            count          mean           std           min           25%  \
WetlandTyp                                                                  
Bog          12.0  6.032309e+04  9.946453e+04   4327.378247  18144.230886   
Fen          96.0  1.148211e+05  1.455728e+05    166.111626  26886.505278   
Forest      274.0  9.382584e+04  1.945735e+05    257.197589  19485.006682   
Marsh         3.0  7.422562e+04  1.835396e+04  53581.055572  66989.664462   
Open water    5.0  4.217620e+06  9.395710e+06   6137.805815  11245.566385   
Shrub        10.0  1.581923e+05  2.268301e+05   1333.360301  16014.101852   
Swamp        42.0  6.365377e+04  6.588182e+04   1434.997905  19823.374517   

                     50%            75%           max  
WetlandTyp                                             
Bog         22836.233729   33547.143322  3.507188e+05  
Fen         57623.624512  161347.927721  8.288974e+05  
Forest      45452.960406   86053.451587  2.109552e+06  
Marsh       80398.273353  

In [6]:
# ============================================
# STEP 4: Generate points on a 100m grid within each training polygon,
# now tagging with WetlandTyp (7 classes) instead of MyClass
# ============================================

GRID_SPACING_M = 100

def grid_points_for_polygon(feature):
    geom = feature.geometry()

    grid_image = ee.Image.pixelCoordinates(ee.Projection('EPSG:32612').atScale(GRID_SPACING_M))

    grid_fc = grid_image.sample(
        region=geom,
        scale=GRID_SPACING_M,
        geometries=True
    )

    grid_fc = grid_fc.map(lambda pt: pt.set({
        'WetlandTyp': feature.get('WetlandTyp'),
        'POLYGON_ID': feature.get('POLYGON_ID')
    }))

    return grid_fc


training_points = training_source.map(grid_points_for_polygon).flatten()

print('Total sample points generated:', training_points.size().getInfo())
print('Class distribution:', training_points.aggregate_histogram('WetlandTyp').getInfo())

Total sample points generated: 6305
Class distribution: {'Bog': 76, 'Fen': 1117, 'Forest': 2568, 'Marsh': 20, 'Open water': 2111, 'Shrub': 156, 'Swamp': 257}
